# FFT radix-2 e Bluestein — Usage e Reference
FFT O(N log N), forward con segno negativo e senza normalizzazione; IFFT con segno positivo e divisione per N. Il notebook usa gli assembly Release, non i pacchetti 0.2.0.
Il risultato chiuso di [1,2,3,4] è [10,-2+2i,-2,-2-2i]: un riferimento indipendente dall'implementazione DFT.

In [ ]:
#load "../shared/MathemaSharp.csx"
using System; using System.Linq; using System.Numerics;
using MathemaSharp.Core;
void Require(bool value,string label) { if(!value) throw new Exception(label); Console.WriteLine(label+": OK"); }
var x=new Complex<double>[] {new(1,0),new(2,0),new(3,0),new(4,0)};
var expected=new Complex<double>[] {new(10,0),new(-2,2),new(-2,0),new(-2,-2)};
var spectrum=Signals.Fft<double>(x);
Require(spectrum.Zip(expected).All(p=>double.Hypot(p.First.Real-p.Second.Real,p.First.Imaginary-p.Second.Imaginary)<1e-13),"closed_form_oracle");
Require(Signals.Ifft<double>(spectrum).Zip(x).All(p=>double.Hypot(p.First.Real-p.Second.Real,p.First.Imaginary-p.Second.Imaginary)<1e-13),"round_trip");
Console.WriteLine(string.Join(", ",spectrum));

closed_form_oracle: OK
round_trip: OK
<10; 0>, <-2; 2>, <-2; 0>, <-1.9999999999999998; -2>

In [ ]:
void Check<T>() where T:IFloatingPointIeee754<T>,IMinMaxValue<T> {
 var a=new Complex<T>[] {new(T.One,T.Zero),new(T.CreateChecked(2),T.Zero),new(T.CreateChecked(3),T.Zero),new(T.CreateChecked(4),T.Zero)};
 var b=Signals.Fft<T>(a);
 var tolerance=(T.BitIncrement(T.One)-T.One)*T.CreateChecked(128);
 Require(T.Abs(b[1].Real-T.CreateChecked(-2))<=tolerance && T.Abs(b[1].Imaginary-T.CreateChecked(2))<=tolerance,typeof(T).Name+"_oracle");
}
Check<Half>(); Check<float>(); Check<double>(); Check<Decimal32>(); Check<Decimal64>(); Check<Decimal128>();
var delta=Decimal128.Parse("1e-25");
var d=Signals.Fft<Decimal128>(new Complex<Decimal128>[] {new(Decimal128.One,Decimal128.Zero),new(delta,Decimal128.Zero)});
Require(d[0].Real-Decimal128.One==delta,"decimal128_increment");

Half_oracle: OK
Single_oracle: OK
Double_oracle: OK
Decimal32_oracle: OK
Decimal64_oracle: OK
Decimal128_oracle: OK
decimal128_increment: OK

## Reference e limiti
| API | Contratto |
|---|---|
| Fft<T> | Input Complex<T>, IFloatingPointIeee754<T> e IMinMaxValue<T> |
| Ifft<T> | Normalizzazione 1/N, non modifica l'input |
| Lunghezza | Qualunque lunghezza, nei limiti di memoria e rappresentabilità di T |
| Valori | Input finiti; risultati non finiti causano ArithmeticException |

Twiddle e accumulo sono calcolati in T. Non è garantita accuratezza universale, né assenza di overflow intermedio. Un piccolo insieme di oracoli non certifica grandi trasformate. Bluestein usa una convoluzione radix-2 con padding >= 2N-1. SIMD e benchmark non sono inclusi.
[Catalogo API](12-api-reference.ipynb).

## Bluestein: lunghezza prima 7
L'impulso ha spettro unitario: riferimento analitico indipendente.

In [ ]:
var prime = new Complex<double>[] { new(1,0), new(0,0), new(0,0), new(0,0), new(0,0), new(0,0), new(0,0) };
var primeSpectrum = Signals.Fft<double>(prime);
Require(primeSpectrum.All(v => double.Hypot(v.Real-1,v.Imaginary)<1e-12), "bluestein_prime_oracle");
var primeBack = Signals.Ifft<double>(primeSpectrum);
Require(primeBack.Zip(prime).All(p => double.Hypot(p.First.Real-p.Second.Real,p.First.Imaginary-p.Second.Imaginary)<1e-12), "bluestein_prime_roundtrip");

bluestein_prime_oracle: OK
bluestein_prime_roundtrip: OK